# 09 - Delegating to Subagents

## Purpose

This notebook demonstrates a LangChain DeepAgents tutorial episode. It walks through the key concepts and code patterns for building, configuring, and operating AI agents with the `deepagents` framework.

## What You'll Learn

- How deep agents differ from regular LangChain agents
- The built-in tool suite every agent receives automatically
- How to configure models, backends, permissions, and memory
- Practical patterns for tools, planning, memory, and handoffs

## Tutorial Series Context

This is Episode 09-subagents_basics of the LangChain DeepAgents YouTube tutorial series. Each episode focuses on a specific aspect of agent development, from basic introduction to advanced patterns like subagents, memory, permissions, and skills.

## How to Use This Notebook

1. **Run cells top-to-bottom**: The code is designed to execute in order, with each section building on the previous one.
2. **Observe tool calls**: Watch for `tool_calls` in message outputs - this shows which agents/tools are being used.
3. **Modify and experiment**: The code is intentionally kept identical to the original Python scripts. Feel free to modify parameters, models, or prompts to see how behavior changes.
4. **Refer to comments**: Each code section has accompanying markdown explanations of what the code does and why.

## Key Concepts Covered in This Episode

- SubAgentMiddleware for task delegation
- Define subagents with name, description, system_prompt
- tools= replaces inherited tool set for sub-agent
- runnable= compiles any LangGraph as a subagent
- Orchestrator only sees final answer, not subagent scratch work
- Context isolation: subagent internal turns never enter history


# Overview: 09-subagents_basics.ipynb

This notebook converts the Python tutorial `09-subagents_basics.ipynb` into an interactive Jupyter Notebook format, preserving all code examples and explanations exactly as they appear in the original tutorial scripts.

## Notebook Structure

Each logical section of the original Python script has been split into its own code cell. Each code cell is preceded by a markdown cell containing:

- **A short heading** describing what that section does
- **1–3 sentences** explaining the purpose and key concepts of that section

## How to Navigate

- **Run cells in order**: The tutorial was designed to be executed top-to-bottom. Each section builds on the previous one.
- **Observe tool calls**: Watch the output for `tool_calls` entries - this shows which built-in or tools the agent invokes.
- **Read the markdown comments**: Each code section has a preceding explanation of what the code does and why it's structured that way.
- **Experiment freely**: The original `.py` files are unchanged. Feel free to modify, extend, or experiment with the code in this notebook.

## Educational Value

This notebook serves as both a working tutorial replication and a learning resource. By running each cell and reading the accompanying explanations, you'll gain hands-on experience with:

- LangChain DeepAgents framework architecture
- Backend configuration (virtual vs real filesystem)
- Tool definition and registration
- Planning, memory, permissions, and context management
- Subagent delegation and structured output
- Streaming modes and human-in-the-loop patterns

## Tips for Getting the Most Out of This Notebook

1. **Don't skip the markdown cells**: The educational content between code sections is essential for understanding the "why" behind the code.
2. **Re-run after modifications**: After experimenting with changes, re-run all cells from the top to ensure everything still works together.
3. **Refer to the original Python scripts**: The `.py` files in the same directory are unchanged and can be referenced for the exact original code.
4. **Use rich output**: This notebook uses Rich formatting for colorful console output - pay attention to color-coded hints and status messages.

---

_Notebook automatically generated from `09-subagents_basics.ipynb`. Original Python scripts preserved unchanged._


## Section 2: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add agent configuration to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for This section sets up the imports and environment configurati
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
"""
09 - Delegating to Subagents
============================
- The `task` tool spawns a subagent that works in an ISOLATED context
- Define one with a `SubAgent` dict: name, description, system_prompt (+ tools)
- The orchestrator only sees the subagent's final answer, not its scratch work
Run:  python 09-subagents_basics.py
"""

## Section 4: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add agent configuration to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for working with DeepAgents
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 1: Imports and setup
import os

from deepagents import create_deep_agent
from dotenv import load_dotenv
from rich import print

load_dotenv()
MODEL = os.getenv("DEEPAGENTS_MODEL", "ollama:qwen3.5:2b")


def text_of(message) -> str:
    """Normalize message content across providers (str vs content blocks)."""
    content = message.content
    if isinstance(content, str):
        return content
    return "".join(block.get("text", "") for block in content if isinstance(block, dict))

## Section 6: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add agent configuration to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for This section shows how to define and delegate tasks to subag
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 2: Define a subagent
# `description` is what the ORCHESTRATOR reads to decide when to delegate —
# write it like a job posting. `system_prompt` is what the SUBAGENT itself
# reads once it is running.
critic_subagent = {
    "name": "critic",
    "description": (
        "Critiques a piece of writing and returns exactly three bullet points "
        "of concrete feedback. Use for any review or critique request."
    ),
    "system_prompt": (
        "You are a blunt writing critic. Given text, reply with exactly three "
        "bullet points of specific, actionable feedback. Nothing else."
    ),
}

## Section 8: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add agent configuration to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for working with DeepAgents
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 3: Give it to the orchestrator
# Every deep agent also gets a built-in "general-purpose" subagent for free;
# yours are added alongside it.
agent = create_deep_agent(
    model=MODEL,
    subagents=[critic_subagent],
    system_prompt=(
        "You are an editor. When the user asks for a critique, you MUST "
        "delegate it to the 'critic' subagent using the task tool rather than "
        "answering yourself. Then summarize its feedback in one sentence."
    ),
)

## Section 10: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add agent configuration to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for working with DeepAgents
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 4: Run a task worth delegating
draft = "Our product is very good and has many features that users like a lot."
task = f"Please critique this sentence using the critic subagent: '{draft}'"
result = agent.invoke({"messages": [{"role": "user", "content": task}]})

## Section 12: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add agent configuration to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for working with DeepAgents
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 5: See the delegation happen
# One `task` call goes out; one consolidated result comes back. The subagent's
# internal turns never enter the orchestrator's message history — that context
# isolation is the whole point.
print("[bold cyan]Delegation:[/bold cyan]")
for message in result["messages"]:
    for call in getattr(message, "tool_calls", []) or []:
        if call["name"] == "task":
            print(f"  -> subagent [green]{call['args'].get('subagent_type')}[/green]")
            print(f"     brief: [dim]{str(call['args'].get('description'))[:80]}[/dim]")

print("\n[bold cyan]What the subagent returned:[/bold cyan]")
for message in result["messages"]:
    if message.__class__.__name__ == "ToolMessage" and message.name == "task":
        print(f"[yellow]{text_of(message).strip()}[/yellow]")

print(f"\n[bold green]Orchestrator's summary:[/bold green] {text_of(result['messages'][-1])}")

# %%